# 🤖 NEW GEN AI Chat Bot
**Author:** Anubhav &nbsp;|&nbsp; **Engine:** Google Gemini 1.5 Flash &nbsp;|&nbsp; **Storage:** JSON Knowledge Base

---

## 📋 Overview
This notebook implements a **dynamically expanding AI chatbot** that:
- Uses **TF-IDF cosine similarity** to search a local JSON knowledge base
- Falls back to **Google Gemini 1.5 Flash** for new questions — and stores the answer
- **Enhances short answers** with Gemini automatically
- **Stores unanswered questions** as pending and resolves them on the next update cycle
- Hosts a **public URL** via `pyngrok` so you can share your chatbot

## 🗂 Files
| File | Purpose |
|---|---|
| `knowledge_base.json` | Q&A store + pending questions |
| `chatbot.py` | Server + Gemini + auto-update logic |
| `index.html` | Simple chat UI |

## 🔧 Setup Steps
1. Install dependencies
2. Add your Gemini API key
3. Upload `knowledge_base.json` and `index.html`
4. Run the server and get a public link
5. Explore visualizations

---
## 1️⃣ Install Dependencies

In [ ]:
# Install required packages
!pip install -q google-generativeai scikit-learn numpy pyngrok
print('✅ All packages installed!')

---
## 2️⃣ Configuration & API Key

In [ ]:
import os
from google.colab import userdata

# ── Option A: Colab Secrets (recommended) ────────────────────────────
# Add your key in: Runtime menu → Manage secrets → Add secret
# Name it: GEMINI_API_KEY
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except Exception:
    # ── Option B: Direct paste (less secure) ─────────────────────────
    GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY_HERE'  # ← paste here
    print('⚠️  Using hardcoded API key — prefer Colab Secrets')

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
print(f'Key preview: {GEMINI_API_KEY[:8]}...')

---
## 3️⃣ Create Knowledge Base (with Pre-installed Q&As)

In [ ]:
import json
from datetime import datetime

KB_FILE = 'knowledge_base.json'

# Pre-installed Q&A pairs covering AI/ML fundamentals
knowledge_base = {
    'metadata': {
        'bot_name':     'NEW GEN AI Chat Bot',
        'author':       'Anubhav',
        'version':      '1.0',
        'last_updated': datetime.now().isoformat(),
        'total_qa':     12
    },
    'qa_pairs': [
        {
            'id': 1, 'question': 'What is artificial intelligence?',
            'answer': 'Artificial Intelligence (AI) simulates human intelligence in machines. It includes learning, reasoning, and self-correction. AI powers voice assistants, recommendation engines, self-driving cars, and medical diagnostics.',
            'tags': ['AI', 'basics'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        },
        {
            'id': 2, 'question': 'What is machine learning?',
            'answer': 'Machine Learning (ML) is a subset of AI that lets systems learn from data without explicit programming. Types: Supervised, Unsupervised, and Reinforcement Learning.',
            'tags': ['ML', 'AI'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        },
        {
            'id': 3, 'question': 'What is deep learning?',
            'answer': 'Deep Learning uses multi-layer neural networks inspired by the human brain. It excels at image recognition, speech processing, and NLP. Frameworks: TensorFlow, PyTorch, Keras.',
            'tags': ['deep learning', 'neural networks'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        },
        {
            'id': 4, 'question': 'What is Python?',
            'answer': 'Python is a high-level, readable programming language created by Guido van Rossum. It is widely used for data science, AI, web development, and automation.',
            'tags': ['Python', 'programming'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        },
        {
            'id': 5, 'question': 'What is Google Gemini?',
            'answer': 'Google Gemini is Google DeepMind\'s most capable multimodal AI model. It understands text, images, audio, video, and code. Available via Google AI Studio API.',
            'tags': ['Gemini', 'Google', 'AI'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        },
        {
            'id': 6, 'question': 'What is a vector database?',
            'answer': 'A vector database stores data as high-dimensional embeddings and enables fast similarity searches. Used in semantic search, recommendation systems, and RAG pipelines.',
            'tags': ['vector database', 'embeddings'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        },
        {
            'id': 7, 'question': 'What is natural language processing?',
            'answer': 'NLP is the AI field focused on understanding and generating human language. Applications: sentiment analysis, translation, chatbots, text summarization.',
            'tags': ['NLP', 'language'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        },
        {
            'id': 8, 'question': 'What is a neural network?',
            'answer': 'A neural network mimics the human brain using layers of nodes (neurons). Input, hidden, and output layers process data through weighted connections trained via backpropagation.',
            'tags': ['neural network', 'deep learning'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        },
        {
            'id': 9, 'question': 'What is the difference between AI and ML?',
            'answer': 'AI is the broad field of making machines intelligent. ML is a subset of AI that uses statistical learning from data. All ML is AI, but not all AI uses ML.',
            'tags': ['AI', 'ML', 'comparison'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        },
        {
            'id': 10, 'question': 'What is data science?',
            'answer': 'Data Science combines statistics, programming, and domain knowledge to extract insights from data. Key tools: Python, Pandas, NumPy, Scikit-learn, Tableau, Jupyter.',
            'tags': ['data science', 'statistics'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        },
        {
            'id': 11, 'question': 'What is a large language model?',
            'answer': 'LLMs are deep learning models trained on massive text corpora with billions of parameters. Examples: GPT-4, Claude, Gemini, LLaMA. They power chatbots, code assistants, and more.',
            'tags': ['LLM', 'language model'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        },
        {
            'id': 12, 'question': 'What is reinforcement learning?',
            'answer': 'Reinforcement Learning trains an agent to maximize reward through environment interaction. Famous uses: AlphaGo, game-playing AIs, robotics, ChatGPT RLHF training.',
            'tags': ['reinforcement learning', 'RL'], 'confidence': 1.0,
            'updated_at': datetime.now().isoformat(), 'asked_count': 0
        }
    ],
    'pending_questions': []
}

with open(KB_FILE, 'w') as f:
    json.dump(knowledge_base, f, indent=2)

print(f'✅ Knowledge base created: {len(knowledge_base["qa_pairs"])} Q&A pairs')
print(f'📄 Saved to: {KB_FILE}')

---
## 4️⃣ Core Chatbot Functions

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai

# ── Gemini setup ──────────────────────────────────────────────────────
genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel('gemini-1.5-flash')

SIMILARITY_TH    = 0.40   # min cosine similarity to consider a KB match
SHORT_ANSWER_LEN = 180    # answers shorter than this get enhanced

# ────────────────────────────────────────────────────────────────────
# KB I/O
# ────────────────────────────────────────────────────────────────────
def load_kb():
    with open(KB_FILE) as f:
        return json.load(f)

def save_kb(kb):
    kb['metadata']['last_updated'] = datetime.now().isoformat()
    kb['metadata']['total_qa']     = len(kb['qa_pairs'])
    with open(KB_FILE, 'w') as f:
        json.dump(kb, f, indent=2)

# ────────────────────────────────────────────────────────────────────
# TF-IDF similarity search
# ────────────────────────────────────────────────────────────────────
def find_best_match(query, kb):
    """Return (best_pair, score) or (None, 0.0)."""
    pairs = kb['qa_pairs']
    if not pairs:
        return None, 0.0
    questions = [p['question'] for p in pairs]
    corpus    = questions + [query]
    tfidf     = TfidfVectorizer(stop_words='english', ngram_range=(1,2)).fit_transform(corpus)
    scores    = cosine_similarity(tfidf[-1], tfidf[:-1]).flatten()
    idx       = int(np.argmax(scores))
    score     = float(scores[idx])
    return (pairs[idx], score) if score >= SIMILARITY_TH else (None, score)

# ────────────────────────────────────────────────────────────────────
# Gemini helpers
# ────────────────────────────────────────────────────────────────────
def call_gemini(prompt):
    try:
        return gemini.generate_content(prompt).text.strip()
    except Exception as e:
        print(f'[Gemini error] {e}')
        return ''

def generate_answer(question):
    return call_gemini(
        'You are NEW GEN AI Chat Bot by Anubhav. '
        'Give a clear, structured, comprehensive answer with bullet points.\n\n'
        f'Question: {question}\n\nAnswer:'
    )

def enhance_answer(question, current):
    return call_gemini(
        'Expand this answer — add detail, bullet points, and examples. '
        'Return only the improved answer.\n\n'
        f'Question: {question}\nCurrent: {current}\n\nEnhanced Answer:'
    )

# ────────────────────────────────────────────────────────────────────
# KB mutation helpers
# ────────────────────────────────────────────────────────────────────
def upsert_qa(kb, question, answer, confidence=1.0):
    """Insert or update a Q&A pair."""
    now = datetime.now().isoformat()
    for p in kb['qa_pairs']:
        if p['question'].lower().strip() == question.lower().strip():
            p.update({'answer': answer, 'confidence': confidence, 'updated_at': now})
            save_kb(kb); return
    new_id = max((p['id'] for p in kb['qa_pairs']), default=0) + 1
    kb['qa_pairs'].append({
        'id': new_id, 'question': question, 'answer': answer,
        'tags': [], 'confidence': confidence,
        'updated_at': now, 'asked_count': 1
    })
    save_kb(kb)

def store_pending(kb, question):
    """Save an unanswered question."""
    now = datetime.now().isoformat()
    for pq in kb['pending_questions']:
        if pq['question'].lower().strip() == question.lower().strip():
            pq['asked_count'] += 1; save_kb(kb); return
    new_id = max((p['id'] for p in kb['pending_questions']), default=0) + 1
    kb['pending_questions'].append({
        'id': new_id, 'question': question, 'answer': None,
        'asked_count': 1, 'first_asked': now, 'status': 'pending'
    })
    save_kb(kb)

# ────────────────────────────────────────────────────────────────────
# Main chat handler
# ────────────────────────────────────────────────────────────────────
def chat(user_message):
    """Process a user message and return a response dict."""
    kb = load_kb()
    match, score = find_best_match(user_message, kb)

    if match:   # ── KB hit ──────────────────────────────────────────
        match['asked_count'] = match.get('asked_count', 0) + 1
        if len(match['answer']) < SHORT_ANSWER_LEN or match.get('confidence', 1) < 0.7:
            enhanced = enhance_answer(match['question'], match['answer'])
            if enhanced:
                match['answer']     = enhanced
                match['confidence'] = 1.0
                match['updated_at'] = datetime.now().isoformat()
        save_kb(kb)
        return {'response': match['answer'], 'source': 'knowledge_base', 'score': round(score,3)}

    answer = generate_answer(user_message)  # ── Gemini call ─────────
    if answer:
        kb = load_kb()
        upsert_qa(kb, user_message, answer)
        return {'response': answer, 'source': 'gemini_new', 'score': round(score,3)}

    kb = load_kb()                           # ── Pending store ───────
    store_pending(kb, user_message)
    return {
        'response': 'Sorry, I don\'t have information on that right now. '
                    'I\'ve saved your question and will update my knowledge base soon! 🔄',
        'source': 'pending', 'score': 0.0
    }

print('✅ Core chatbot functions loaded!')
print('Functions: chat(), find_best_match(), upsert_qa(), store_pending()')

---
## 5️⃣ Quick Test — Console Chat

In [ ]:
# Test with pre-installed questions
test_questions = [
    'What is artificial intelligence?',
    'Tell me about machine learning',
    'Explain deep learning to me',
    'What is the latest iPhone model?',   # <-- likely NOT in KB, tests Gemini fallback
]

print('=' * 60)
print('   🤖  NEW GEN AI Chat Bot  |  Test Session')
print('=' * 60)

for q in test_questions:
    print(f'\n👤 USER: {q}')
    result = chat(q)
    # Truncate long responses for display
    resp = result['response']
    if len(resp) > 300:
        resp = resp[:300] + '...'
    src_icons = {'knowledge_base': '📚', 'gemini_new': '✨', 'pending': '⏳'}
    icon = src_icons.get(result['source'], '🤖')
    print(f'🤖 BOT [{icon} {result["source"]} | score={result["score"]}]: {resp}')
    print('-' * 60)

---
## 6️⃣ 📊 Visualizations — Knowledge Base Analytics

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
from collections import Counter

sns.set_theme(style='dark', palette='muted')
plt.rcParams.update({
    'figure.facecolor': '#0d1424', 'axes.facecolor': '#080c14',
    'text.color': '#e2e8f0', 'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
    'axes.edgecolor': '#1a2540', 'grid.color': '#1a2540'
})

kb = load_kb()
pairs = kb['qa_pairs']
df = pd.DataFrame(pairs)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('🤖 NEW GEN AI Chat Bot — Knowledge Base Analytics', 
             fontsize=16, fontweight='bold', color='#00d4ff', y=1.01)

# ── Plot 1: Answer length distribution ─────────────────────────────
ax1 = axes[0, 0]
lengths = df['answer'].str.len()
ax1.hist(lengths, bins=12, color='#00d4ff', alpha=0.8, edgecolor='#080c14')
ax1.axvline(lengths.mean(), color='#f59e0b', lw=2, linestyle='--', label=f'Mean={lengths.mean():.0f}')
ax1.set_title('Answer Length Distribution', color='#e2e8f0', pad=10)
ax1.set_xlabel('Characters')
ax1.set_ylabel('Count')
ax1.legend(facecolor='#0d1424', labelcolor='#e2e8f0')
ax1.grid(True, alpha=0.3)

# ── Plot 2: Tag frequency ───────────────────────────────────────────
ax2 = axes[0, 1]
all_tags = [tag for pair in pairs for tag in pair.get('tags', [])]
if all_tags:
    tag_counts = Counter(all_tags).most_common(10)
    tags, counts = zip(*tag_counts)
    colors = plt.cm.cool(np.linspace(0.2, 0.9, len(tags)))  # type: ignore
    bars = ax2.barh(list(tags), list(counts), color=colors)
    ax2.set_title('Top Tags in Knowledge Base', color='#e2e8f0', pad=10)
    ax2.set_xlabel('Frequency')
    ax2.grid(True, alpha=0.3, axis='x')
    for bar, count in zip(bars, counts):
        ax2.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 str(count), va='center', color='#e2e8f0', fontsize=9)

# ── Plot 3: Confidence score distribution ──────────────────────────
ax3 = axes[1, 0]
confidence_vals = df['confidence'].value_counts()
pie_colors = ['#10b981', '#f59e0b', '#ef4444']
ax3.pie(confidence_vals.values, labels=[f'Conf={v}' for v in confidence_vals.index],
        colors=pie_colors[:len(confidence_vals)], autopct='%1.0f%%',
        textprops={'color': '#e2e8f0'}, startangle=90)
ax3.set_title('Confidence Score Distribution', color='#e2e8f0', pad=10)

# ── Plot 4: KB growth simulation ───────────────────────────────────
ax4 = axes[1, 1]
days = list(range(1, 31))
# Simulate exponential-ish KB growth
np.random.seed(42)
base = len(pairs)
growth = [base + int(np.random.poisson(1.5) * d) for d in days]
ax4.fill_between(days, growth, alpha=0.25, color='#7c3aed')
ax4.plot(days, growth, color='#a78bfa', lw=2.5, marker='o', markersize=4)
ax4.set_title('Simulated KB Growth Over 30 Days', color='#e2e8f0', pad=10)
ax4.set_xlabel('Day')
ax4.set_ylabel('Total Q&A Pairs')
ax4.grid(True, alpha=0.3)
ax4.axhline(base, color='#00d4ff', lw=1.5, linestyle='--', label=f'Start: {base}')
ax4.legend(facecolor='#0d1424', labelcolor='#e2e8f0')

plt.tight_layout()
plt.savefig('kb_analytics.png', dpi=150, bbox_inches='tight', facecolor='#0d1424')
plt.show()
print('\n📊 Analytics chart saved to kb_analytics.png')

In [ ]:
# ── Similarity matrix heatmap ──────────────────────────────────────
kb = load_kb()
questions = [p['question'] for p in kb['qa_pairs']]
short_labels = [q[:30] + '...' if len(q) > 30 else q for q in questions]

tfidf_mat = TfidfVectorizer(stop_words='english').fit_transform(questions)
sim_matrix = cosine_similarity(tfidf_mat).round(2)

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor('#080c14')
ax.set_facecolor('#080c14')

mask = np.eye(len(questions), dtype=bool)   # hide diagonal (self-similarity=1)
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(
    sim_matrix, mask=mask, cmap='YlOrRd', vmin=0, vmax=0.6,
    xticklabels=short_labels, yticklabels=short_labels,
    linewidths=0.4, linecolor='#1a2540', annot=True, fmt='.2f',
    annot_kws={'size': 6, 'color': '#e2e8f0'},
    cbar_kws={'shrink': .7}, ax=ax
)
ax.set_title('Question Similarity Matrix (TF-IDF Cosine)', 
             fontsize=13, color='#00d4ff', pad=14)
plt.xticks(rotation=45, ha='right', fontsize=7, color='#94a3b8')
plt.yticks(rotation=0, fontsize=7, color='#94a3b8')
plt.tight_layout()
plt.savefig('similarity_matrix.png', dpi=120, bbox_inches='tight', facecolor='#080c14')
plt.show()
print('\n🗺️  Similarity heatmap saved to similarity_matrix.png')

---
## 7️⃣ Create the Simple HTML Frontend

In [ ]:
# Write the minimal HTML UI file
html_content = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <title>NEW GEN AI Chat Bot</title>
  <link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;800&family=Space+Mono&display=swap" rel="stylesheet">
  <style>
    *{box-sizing:border-box;margin:0;padding:0}
    :root{--bg:#080c14;--panel:#0d1424;--border:#1a2540;--accent:#00d4ff;--accent2:#7c3aed;--text:#e2e8f0;--muted:#64748b}
    body{font-family:Syne,sans-serif;background:var(--bg);color:var(--text);height:100vh;display:grid;place-items:center}
    .app{display:grid;grid-template-rows:auto 1fr auto;height:100vh;width:min(860px,100%);padding:0 12px}
    header{padding:20px 0 12px;text-align:center;border-bottom:1px solid var(--border)}
    header h1{font-size:clamp(1.4rem,4vw,2rem);font-weight:800;background:linear-gradient(135deg,var(--accent),#a78bfa,var(--accent2));-webkit-background-clip:text;-webkit-text-fill-color:transparent;background-clip:text}
    .meta{font-family:"Space Mono",monospace;font-size:.65rem;color:var(--muted);margin-top:4px}
    .chat{overflow-y:auto;padding:16px 0;display:flex;flex-direction:column;gap:12px;scroll-behavior:smooth}
    .chat::-webkit-scrollbar{width:3px}.chat::-webkit-scrollbar-thumb{background:var(--border);border-radius:3px}
    .msg{display:flex;gap:8px;align-items:flex-start;animation:pop .2s ease-out}
    @keyframes pop{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:translateY(0)}}
    .msg.user{flex-direction:row-reverse}
    .av{width:32px;height:32px;border-radius:8px;display:flex;align-items:center;justify-content:center;font-size:.9rem;flex-shrink:0}
    .msg.bot .av{background:#111927;border:1px solid var(--accent)}
    .msg.user .av{background:#1a0d40;border:1px solid var(--accent2)}
    .bbl{max-width:75%;padding:10px 14px;border-radius:12px;font-size:.87rem;line-height:1.65;white-space:pre-wrap;word-break:break-word}
    .msg.bot .bbl{background:#111927;border:1px solid var(--border);border-top-left-radius:3px}
    .msg.user .bbl{background:#1a0d40;border:1px solid rgba(124,58,237,.35);border-top-right-radius:3px}
    .input-area{padding:12px 0 18px;border-top:1px solid var(--border)}
    .row{display:flex;gap:8px;align-items:flex-end;background:var(--panel);border:1px solid var(--border);border-radius:14px;padding:8px 12px;transition:border-color .2s}
    .row:focus-within{border-color:var(--accent)}
    textarea{flex:1;background:transparent;border:none;outline:none;color:var(--text);font-family:Syne,sans-serif;font-size:.88rem;resize:none;max-height:100px;line-height:1.5}
    textarea::placeholder{color:var(--muted)}
    button{width:36px;height:36px;border-radius:9px;border:none;background:linear-gradient(135deg,var(--accent),var(--accent2));color:#fff;cursor:pointer;font-size:1rem;flex-shrink:0;transition:.15s}
    button:hover{opacity:.85}button:active{transform:scale(.92)}button:disabled{opacity:.35;cursor:not-allowed}
    .hint{text-align:center;font-size:.6rem;color:var(--muted);margin-top:6px;font-family:"Space Mono",monospace}
    .dot{width:5px;height:5px;border-radius:50%;background:var(--accent);animation:b 1.2s infinite}
    .dot:nth-child(2){animation-delay:.2s}.dot:nth-child(3){animation-delay:.4s}
    @keyframes b{0%,60%,100%{transform:translateY(0);opacity:.4}30%{transform:translateY(-5px);opacity:1}}
  </style>
</head>
<body>
<div class="app">
  <header>
    <h1>I am Here to help you.</h1>
    <p class="meta">NEW GEN AI Chat Bot &nbsp;·&nbsp; by Anubhav &nbsp;·&nbsp; Powered by Google Gemini</p>
  </header>
  <div class="chat" id="chat">
    <div class="msg bot"><div class="av">🤖</div><div class="bbl">Hello! I am NEW GEN AI Chat Bot by Anubhav. Ask me anything about AI, technology, or science — I learn as we chat! 🚀</div></div>
  </div>
  <div class="input-area">
    <div class="row">
      <textarea id="inp" rows="1" placeholder="Ask me anything…"></textarea>
      <button id="btn" onclick="send()">&#9658;</button>
    </div>
    <p class="hint">Enter to send · Shift+Enter for new line · Auto-learns from every question 🔄</p>
  </div>
</div>
<script>
  const inp=document.getElementById("inp"),btn=document.getElementById("btn"),chat=document.getElementById("chat");
  inp.addEventListener("input",()=>{inp.style.height="auto";inp.style.height=Math.min(inp.scrollHeight,100)+"px"});
  inp.addEventListener("keydown",e=>{if(e.key==="Enter"&&!e.shiftKey){e.preventDefault();send();}});
  function esc(s){return s.replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;");}
  function md(t){return t.replace(/\*\*(.+?)\*\*/g,"<strong>$1</strong>").replace(/\*(.+?)\*/g,"<em>$1</em>").replace(/^- (.+)/gm,"• $1").replace(/\n/g,"<br>");}
  function addMsg(role,html){const d=document.createElement("div");d.className="msg "+role;d.innerHTML=`<div class="av">${role==="bot"?"🤖":"🧑"}</div><div class="bbl">${html}</div>`;chat.appendChild(d);chat.scrollTop=chat.scrollHeight;return d;}
  function typing(){const d=document.createElement("div");d.className="msg bot";d.id="typ";d.innerHTML=`<div class="av">🤖</div><div class="bbl" style="display:flex;gap:5px;align-items:center"><div class="dot"></div><div class="dot"></div><div class="dot"></div></div>`;chat.appendChild(d);chat.scrollTop=chat.scrollHeight;}
  async function send(){const msg=inp.value.trim();if(!msg)return;addMsg("user",esc(msg));inp.value="";inp.style.height="auto";btn.disabled=true;typing();
    try{const r=await fetch("/chat",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({message:msg})});const d=await r.json();document.getElementById("typ").remove();addMsg("bot",md(d.response||"No response."));}catch{document.getElementById("typ")?.remove();addMsg("bot","⚠️ Server unreachable.");}finally{btn.disabled=false;inp.focus();}}
</script>
</body></html>'''

with open('index.html', 'w') as f:
    f.write(html_content)

print('✅ index.html created!')

---
## 8️⃣ Start Server + Get Public URL (ngrok)

In [ ]:
# ─── Write the full server file ────────────────────────────────────
server_code = '''
import json, os, time, threading, numpy as np
from datetime import datetime
from http.server import HTTPServer, BaseHTTPRequestHandler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
KB_FILE = "knowledge_base.json"
SIMILARITY_TH = 0.40
SHORT_ANSWER_LEN = 180
UPDATE_INTERVAL = 1800
PORT = 8080

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel("gemini-1.5-flash")

def load_kb():
    with open(KB_FILE) as f: return json.load(f)
def save_kb(kb):
    kb["metadata"]["last_updated"] = datetime.now().isoformat()
    kb["metadata"]["total_qa"] = len(kb["qa_pairs"])
    with open(KB_FILE, "w") as f: json.dump(kb, f, indent=2)

def find_best_match(query, kb):
    pairs = kb["qa_pairs"]
    if not pairs: return None, 0.0
    questions = [p["question"] for p in pairs]
    tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1,2)).fit_transform(questions + [query])
    scores = cosine_similarity(tfidf[-1], tfidf[:-1]).flatten()
    idx = int(np.argmax(scores)); score = float(scores[idx])
    return (pairs[idx], score) if score >= SIMILARITY_TH else (None, score)

def call_gemini(prompt):
    try: return gemini.generate_content(prompt).text.strip()
    except Exception as e: print(f"Gemini error: {e}"); return ""

def generate_answer(q): return call_gemini(f"You are NEW GEN AI Chat Bot by Anubhav. Answer clearly with bullet points.\n\nQuestion: {q}\n\nAnswer:")
def enhance_answer(q, a): return call_gemini(f"Expand this answer with detail and bullet points. Return only the enhanced answer.\n\nQ: {q}\nCurrent: {a}\n\nEnhanced:")

def upsert_qa(kb, question, answer, confidence=1.0):
    now = datetime.now().isoformat()
    for p in kb["qa_pairs"]:
        if p["question"].lower().strip() == question.lower().strip():
            p.update({"answer": answer, "confidence": confidence, "updated_at": now}); save_kb(kb); return
    new_id = max((p["id"] for p in kb["qa_pairs"]), default=0) + 1
    kb["qa_pairs"].append({"id": new_id, "question": question, "answer": answer, "tags": [], "confidence": confidence, "updated_at": now, "asked_count": 1})
    save_kb(kb)

def store_pending(kb, question):
    now = datetime.now().isoformat()
    for pq in kb["pending_questions"]:
        if pq["question"].lower().strip() == question.lower().strip():
            pq["asked_count"] += 1; save_kb(kb); return
    new_id = max((p["id"] for p in kb["pending_questions"]), default=0) + 1
    kb["pending_questions"].append({"id": new_id, "question": question, "answer": None, "asked_count": 1, "first_asked": now, "status": "pending"})
    save_kb(kb)

def auto_update_loop():
    while True:
        time.sleep(UPDATE_INTERVAL)
        try:
            kb = load_kb()
            pending = sorted([p for p in kb["pending_questions"] if p["status"]=="pending"], key=lambda x: x["asked_count"], reverse=True)
            for pq in pending[:5]:
                ans = generate_answer(pq["question"])
                if ans: upsert_qa(kb, pq["question"], ans); pq["status"]="resolved"; kb=load_kb()
            save_kb(kb)
            kb = load_kb()
            for pair in kb["qa_pairs"]:
                if len(pair.get("answer","")) < 180 or pair.get("confidence",1) < 0.7:
                    enh = enhance_answer(pair["question"], pair["answer"])
                    if enh: pair["answer"]=enh; pair["confidence"]=1.0; pair["updated_at"]=datetime.now().isoformat()
            save_kb(kb)
        except Exception as e: print(f"Auto-update error: {e}")

def process_message(msg):
    kb = load_kb(); match, score = find_best_match(msg, kb)
    if match:
        match["asked_count"] = match.get("asked_count",0)+1
        if len(match["answer"]) < 180 or match.get("confidence",1) < 0.7:
            enh = enhance_answer(match["question"], match["answer"])
            if enh: match["answer"]=enh; match["confidence"]=1.0; match["updated_at"]=datetime.now().isoformat()
        save_kb(kb)
        return {"response": match["answer"], "source": "knowledge_base", "score": round(score,3)}
    ans = generate_answer(msg)
    if ans:
        kb = load_kb(); upsert_qa(kb, msg, ans)
        return {"response": ans, "source": "gemini_new", "score": round(score,3)}
    kb = load_kb(); store_pending(kb, msg)
    return {"response": "Sorry, I don't have info on that right now. I've saved your question and will update soon! 🔄", "source": "pending", "score": 0.0}

class ChatHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args): print(f"[{datetime.now().strftime('%H:%M:%S')}] {fmt % args}")
    def _cors(self):
        self.send_header("Access-Control-Allow-Origin","*")
        self.send_header("Access-Control-Allow-Methods","GET,POST,OPTIONS")
        self.send_header("Access-Control-Allow-Headers","Content-Type")
    def _json(self, data, status=200):
        body = json.dumps(data, indent=2).encode()
        self.send_response(status); self.send_header("Content-Type","application/json"); self._cors(); self.end_headers(); self.wfile.write(body)
    def do_GET(self):
        if self.path in ("/","/index.html"):
            with open("index.html","rb") as f: content=f.read()
            self.send_response(200); self.send_header("Content-Type","text/html"); self._cors(); self.end_headers(); self.wfile.write(content)
        elif self.path=="/stats": self._json({"total_qa":len(load_kb()["qa_pairs"]),"pending":len([p for p in load_kb()["pending_questions"] if p["status"]=="pending"])})
        elif self.path=="/kb": self._json(load_kb())
        else: self.send_response(404); self.end_headers()
    def do_POST(self):
        if self.path=="/chat":
            length=int(self.headers.get("Content-Length",0))
            body=json.loads(self.rfile.read(length))
            msg=body.get("message","").strip()
            if not msg: self._json({"error":"Empty message"},400); return
            self._json(process_message(msg))
        else: self.send_response(404); self.end_headers()
    def do_OPTIONS(self): self.send_response(200); self._cors(); self.end_headers()

if __name__ == "__main__":
    threading.Thread(target=auto_update_loop, daemon=True).start()
    print(f"Server running on port {PORT}")
    HTTPServer(("0.0.0.0", PORT), ChatHandler).serve_forever()
'''

with open('chatbot.py', 'w') as f:
    f.write(server_code)

print('✅ chatbot.py written!')

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok, conf

PORT = 8080

# ── Optional: set your ngrok auth token (get free at ngrok.com) ─────
# NGROK_TOKEN = userdata.get('NGROK_TOKEN')  # or paste directly
# conf.get_default().auth_token = NGROK_TOKEN

# ── Start server subprocess ──────────────────────────────────────────
proc = subprocess.Popen(['python', 'chatbot.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(3)   # wait for server to initialize

# ── Open ngrok tunnel ────────────────────────────────────────────────
public_url = ngrok.connect(PORT).public_url

print('=' * 55)
print('   🤖  NEW GEN AI Chat Bot  |  by Anubhav')
print('=' * 55)
print(f'   🌐  Public URL : {public_url}')
print(f'   📚  KB endpoint: {public_url}/kb')
print(f'   📊  Stats      : {public_url}/stats')
print('=' * 55)
print('   Share the public URL with anyone!')

---
## 9️⃣ Live KB Dashboard

In [ ]:
# Display a formatted summary of the knowledge base
kb = load_kb()

print(f"""{'='*60}
  🤖  {kb['metadata']['bot_name']}
  👤  Author   : {kb['metadata']['author']}
  📅  Updated  : {kb['metadata']['last_updated'][:19]}
  📚  Total QA : {len(kb['qa_pairs'])}
  ⏳  Pending  : {len([p for p in kb['pending_questions'] if p['status']=='pending'])}
  ✅  Resolved : {len([p for p in kb['pending_questions'] if p['status']=='resolved'])}
{'='*60}""")

print("\n📋 Knowledge Base Entries:\n")
for i, pair in enumerate(kb['qa_pairs'], 1):
    ans_preview = pair['answer'][:80] + '...' if len(pair['answer']) > 80 else pair['answer']
    print(f"  [{i:02d}] Q: {pair['question']}")
    print(f"       A: {ans_preview}")
    print(f"       🔒 conf={pair['confidence']} | asked={pair['asked_count']} | tags={pair['tags']}\n")

if kb['pending_questions']:
    print("\n⏳ Pending Questions:\n")
    for pq in kb['pending_questions']:
        print(f"  [{pq['status'].upper()}] {pq['question']} (asked {pq['asked_count']}x)")

---
## 🛑 Stop Server

In [ ]:
# Run this cell to stop the server and close ngrok tunnel
ngrok.disconnect(public_url)
proc.terminate()
print('✅ Server stopped and ngrok tunnel closed.')

---
## 📌 Notes & Next Steps

- **API Key**: Get your free Gemini API key at [aistudio.google.com](https://aistudio.google.com)
- **ngrok Token**: Get a free token at [ngrok.com](https://ngrok.com) for reliable tunneling
- **Customize**: Edit `SIMILARITY_TH` in `chatbot.py` to tune KB matching strictness
- **Add Sources**: Call `upsert_qa(kb, question, answer)` to manually seed the knowledge base
- **Production**: Deploy `chatbot.py` to a VPS/cloud instance for 24/7 availability

---
*Created by **Anubhav** · NEW GEN AI Chat Bot · Google Gemini 1.5 Flash*